In [ ]:
import pandas as pd

df = pd.read_csv(
    "../data/processed_career_dataset.csv"
)

print(df.shape)

df.head()

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    stop_words='english'
)

tfidf_matrix = tfidf.fit_transform(
    df['student_profile']
)

print(tfidf_matrix.shape)

from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(
    tfidf_matrix
)

print(similarity_matrix.shape)
def recommend_career(student_index):

    similarity_scores = list(
        enumerate(
            similarity_matrix[student_index]
        )
    )

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    careers = []

    for idx, score in similarity_scores[1:]:

        career = df.iloc[idx]['Career']

        if career not in careers:

            careers.append(career)

        if len(careers) == 3:

            break

    return careers
recommend_career(0)
def recommend_with_score(student_index):

    similarity_scores = list(
        enumerate(
            similarity_matrix[student_index]
        )
    )

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    recommendations = []

    used = set()

    for idx, score in similarity_scores[1:]:

        career = df.iloc[idx]['Career']

        if career not in used:

            recommendations.append(
                (
                    career,
                    round(score * 100, 2)
                )
            )

            used.add(career)

        if len(recommendations) == 3:

            break

    return recommendations
recommend_with_score(0)
def recommend_from_input(
    skills,
    interest
):

    user_profile = (
        str(skills) + " " +
        str(interest)
    )

    user_vector = tfidf.transform(
        [user_profile]
    )

    similarity_scores = cosine_similarity(
        user_vector,
        tfidf_matrix
    )

    similarity_scores = list(
        enumerate(
            similarity_scores[0]
        )
    )

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    recommendations = []

    used = set()

    for idx, score in similarity_scores:

        career = df.iloc[idx]['Career']

        if career not in used:

            recommendations.append(
                (
                    career,
                    float(round(score * 100, 2))
                )
            )

            used.add(career)

        if len(recommendations) == 3:

            break

    return recommendations
recommend_from_input(
    skills="Python, SQL",
    interest="AI/ML"
)
import joblib

import os

BASE_DIR = os.path.dirname(
    os.path.dirname(
        os.path.abspath(__file__)
    )
)

MODEL_PATH = os.path.join(
    BASE_DIR,
    "models",
    "tfidf.pkl"
)

tfidf = joblib.load(MODEL_PATH)

joblib.dump(
    tfidf_matrix,
    "../models/tfidf_matrix.pkl"
)

print("TF-IDF Model Saved")


(15895, 20)
(15895, 53)
(15895, 15895)
TF-IDF Model Saved
